# PPR10K Setup — 解压版 (for ECCV rebuttal)

**前提**: 你已经把 PPR10K 的官方 [Drive 文件夹 1kB2OSA...](https://drive.google.com/drive/folders/1kB2OSAGy8uc0xUXaMKoPB0HMSc-rkrLW) 加 shortcut 到 `MyDrive/datasets/`。所以 Colab 看到的路径是:
```
/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p/
    source.zip          12 GB   ← 输入(含 5x 增强,我们只用无增强版)
    target_a.zip         4.6 GB  ← expert a 输出
    target_b.zip         4.5 GB
    target_c.zip         4.7 GB
```

**策略**: 把 source + target_a (~17GB) 解压到 Colab `/content/ppr10k/`,**不要解到 Drive**(Drive 写小文件慢、训练读小文件也慢)。每次 Colab 会话重启后重解一次,大概 3-5 分钟,远比训练时读 Drive 快。

**输出**: 训练数据放 `/content/ppr10k/paired_a/{train,val}/{input,gt}/`,checkpoints 仍存到 Drive 持久化。

In [ ]:
# === Cell: session start — mount Drive + pull latest code ===
# Drive 这份 LoR-LUT 是只读副本,只接收 git pull,从不 commit。
# 跑 cell 时 Colab 会自动把 outputs 写进 .ipynb,git 会看成 'modified'。
# 所以 pull 前先 checkout notebooks/ 丢掉那些自动写入,working tree 才干净。
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/LoR-LUT

# 丢掉 Colab 自动改的 ipynb outputs (永远 OK,我们的 ipynb outputs 不重要)
!git checkout -- notebooks/

# 拉最新代码 (--ff-only: 万一 Drive 那份意外有 commit,立即 abort 不悄悄 merge)
!git pull --ff-only origin main

# 确认拿到最新
!git log --oneline -3

In [ ]:
# === Cell 1: mount + 验证 shortcut 在哪 ===
from google.colab import drive
drive.mount('/content/drive')

import os
PPR_360P = '/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p'
assert os.path.exists(PPR_360P), f"❌ {PPR_360P} 不存在 — 检查 shortcut 是否加在了 MyDrive/datasets/ 下"

print("✅ 找到 PPR10K 360p 目录")
for f in sorted(os.listdir(PPR_360P)):
    full = f'{PPR_360P}/{f}'
    if os.path.isfile(full):
        sz_gb = os.path.getsize(full) / 1e9
        print(f"  📄 {f} ({sz_gb:.2f} GB)")
    else:
        print(f"  📂 {f}/")

In [ ]:
# === Cell 2: 解压 source.zip + source_aug_6 (multi-part) + target_a.zip 到 /content/ ===
# 比之前多一份 source_aug_6 (5x augmented 训练数据). 总共 ~30GB,Colab /content 够用.
EXPERT = 'a'
WORK = '/content/ppr10k_raw'

import os, time
os.makedirs(WORK, exist_ok=True)

# 装 7z (有时 Colab 默认没装)
!apt-get -qq install -y p7zip-full > /dev/null

# (1) 单 zip: source.zip 和 target_a.zip
for zip_name in ['source.zip', f'target_{EXPERT}.zip']:
    src_path = f'{PPR_360P}/{zip_name}'
    out_dir = f'{WORK}/{zip_name.replace(".zip", "")}'
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 100:
        print(f'⏭️  {zip_name} 已解 ({len(os.listdir(out_dir))} 文件)')
        continue
    print(f'📦 解 {zip_name} ({os.path.getsize(src_path)/1e9:.2f} GB)...')
    t0 = time.time()
    !unzip -q -o '{src_path}' -d {WORK}/
    print(f'   ✅ {time.time()-t0:.0f}s')

# (2) Multi-part zip: source_aug_6.zip.001..007
AUG_DIR = f'{WORK}/source_aug_6'
if os.path.exists(AUG_DIR) and len(os.listdir(AUG_DIR)) > 100:
    print(f'⏭️  source_aug_6 已解 ({len(os.listdir(AUG_DIR))} 文件)')
else:
    parts_dir = f'{PPR_360P}/source_aug_6_zip'
    parts = sorted([f for f in os.listdir(parts_dir) if f.startswith('source_aug_6.zip.')])
    total_gb = sum(os.path.getsize(f'{parts_dir}/{p}') for p in parts) / 1e9
    print(f'📦 解 source_aug_6 multi-part ({len(parts)} parts, {total_gb:.2f} GB)...')
    t0 = time.time()
    # 7z 直接处理 multi-part split zip,指向 .001 即可
    first_part = f'{parts_dir}/{parts[0]}'
    !cd {WORK} && 7z x -bd -y '{first_part}' > /tmp/7z.log 2>&1 && echo '7z OK' || (echo '7z 失败,日志:'; tail -20 /tmp/7z.log)
    print(f'   ✅ {time.time()-t0:.0f}s')

# 看结构
print('\n=== /content/ppr10k_raw/ 内容 ===')
!ls -la {WORK}/
for d in sorted(os.listdir(WORK)):
    full = f'{WORK}/{d}'
    if os.path.isdir(full):
        files = sorted(os.listdir(full))
        print(f'\n📂 {d}/ ({len(files)} 个文件)')
        print(f'   头 5: {files[:5]}')
        print(f'   尾 3: {files[-3:]}')

In [ ]:
# === Cell 3: 整理 paired 结构 (含 augmented sources) ===
# 原始 source: <group>_<photo>.tif         → 配 target_a/<group>_<photo>.tif
# 增强 source: <group>_<photo>_<aug>.tif   → 也配 target_a/<group>_<photo>.tif (同一个 target)
#
# 每个 augmented input 在 train/input/ 里用唯一文件名存 (symlink),对应的 target
# 在 train/gt/ 里用 SAME 唯一文件名 link 到 target_a/<group>_<photo>.tif
# 这样 paired_folder.py (按文件名匹配 input/gt) 能正确配对.
import os, re

SOURCE_DIR     = f'{WORK}/source'           # 11,161 原始
SOURCE_AUG_DIR = f'{WORK}/source_aug_6'     # ~5 倍增强
TARGET_DIR     = f'{WORK}/target_{EXPERT}'  # 11,161 target
DEST = f'/content/ppr10k_paired_{EXPERT}_aug'  # 注意目录名带 _aug 区别非增强版

for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    os.makedirs(f'{DEST}/{sub}', exist_ok=True)

# 看一眼 augmented 的实际命名 (debug)
if os.path.exists(SOURCE_AUG_DIR):
    aug_sample = sorted(os.listdir(SOURCE_AUG_DIR))[:8]
    print(f'📂 source_aug_6/ 头 8 个: {aug_sample}')

tgt_set = set(os.listdir(TARGET_DIR))

def parse_target_id(fname):
    """提取 <group>_<photo> 用于切 train/val"""
    m = re.match(r'^(\d+)_(\d+)\.tif$', fname)
    return (int(m.group(1)), int(m.group(2))) if m else None

def find_target(source_fname, tgt_set):
    """给定 source 文件名,找对应的 target.
    PPR10K 命名约定:
      原始 source: <group>_<photo>.tif        (如 0_0.tif)
      增强 source: <group>_<photo>_<aug>.tif  (如 0_0_1.tif, aug 1-5)
    都对应同一个 target_a/<group>_<photo>.tif"""
    # 原始 source
    m1 = re.match(r'^(\d+)_(\d+)\.tif$', source_fname)
    if m1 and source_fname in tgt_set:
        return source_fname, (int(m1.group(1)), int(m1.group(2)))
    # 增强 source: 去掉最后一个 _<digits> suffix 得到 target 名
    m2 = re.match(r'^(\d+)_(\d+)_(\d+)\.tif$', source_fname)
    if m2:
        target = f'{m2.group(1)}_{m2.group(2)}.tif'
        if target in tgt_set:
            return target, (int(m2.group(1)), int(m2.group(2)))
    return None, None

# 准备 unified source list: 原始 + 增强
all_sources = []
for f in sorted(os.listdir(SOURCE_DIR)):
    all_sources.append((SOURCE_DIR, f))
if os.path.exists(SOURCE_AUG_DIR):
    for f in sorted(os.listdir(SOURCE_AUG_DIR)):
        all_sources.append((SOURCE_AUG_DIR, f))
print(f'总 source 数 (原+增强): {len(all_sources)}')

# 配对 + split
TRAIN_GROUP_CUTOFF = 1345  # 0-1344 训练 / 1345-1680 验证
n_train, n_val, n_skip = 0, 0, 0
no_match_samples = []

def link(src, dst):
    if not os.path.exists(dst):
        try: os.symlink(src, dst)
        except FileExistsError: pass

for src_dir, sf in all_sources:
    target_fname, gid_pid = find_target(sf, tgt_set)
    if target_fname is None:
        n_skip += 1
        if len(no_match_samples) < 5:
            no_match_samples.append(sf)
        continue
    
    group_id = gid_pid[0]
    split = 'train' if group_id < TRAIN_GROUP_CUTOFF else 'val'
    
    # val 只用原始 source (不用增强,跟 paper 一致)
    if split == 'val' and src_dir == SOURCE_AUG_DIR:
        continue
    
    link(f'{src_dir}/{sf}', f'{DEST}/{split}/input/{sf}')
    link(f'{TARGET_DIR}/{target_fname}', f'{DEST}/{split}/gt/{sf}')
    if split == 'train':
        n_train += 1
    else:
        n_val += 1

print(f'\n配对结果: train={n_train}, val={n_val}, skipped={n_skip}')
if no_match_samples:
    print(f'⚠️ 配对失败样本 (前 5): {no_match_samples}')
    print('   如果有失败,augmented 命名 pattern 跟我猜的不一样,需要调 find_target() 里的 candidates')

print(f'\n✅ {DEST}')
for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    print(f"   {sub}: {len(os.listdir(f'{DEST}/{sub}'))} 文件")

In [ ]:
# === Cell 4: 验证 paired_folder 能加载,且 input != gt ===
import sys
sys.path.insert(0, '/content/drive/MyDrive/LoR-LUT')
from data.paired_folder import PairedFolderDataset

ds = PairedFolderDataset(
    root=DEST,
    split='train',
    in_dir='input',
    gt_dir='gt',
    exts=('.tif', '.tiff'),
    patch=0,
    augment=False
)
print(f"✅ {len(ds)} train pairs")
for i in [0, 1, 2, len(ds)//2, len(ds)-1]:
    s = ds[i]
    diff = (s['img_in'] - s['img_gt']).abs().mean().item()
    status = '✅' if diff > 0.01 else '⚠️ input==gt!'
    print(f"  [{i:5d}] {s['name']:20s} input-gt MAE={diff:.4f} {status}")

In [ ]:
# === Cell 5: 训练命令 (Colab /content 数据 + Drive ckpt) ===
import datetime
EXP_NAME = f"ppr10k_{EXPERT}_K0_R8_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"

%cd /content/drive/MyDrive/LoR-LUT

!python train.py \
    --cfg config/default.yaml \
    --data.root /content/ppr10k_paired_{EXPERT}_aug \
    --work_dir /content/drive/MyDrive/LoR-LUT/runs/{EXP_NAME}

## 故障排除

**Cell 2 解压超慢/卡住** —— 检查 Colab 磁盘剩余 (`!df -h /content`)。如果 < 30GB 空间,先 `!rm -rf /content/sample_data` 清出空间。

**Cell 3 配对数 < 100** —— PPR10K 命名跟我猜的不一样。运行下面看实际命名,然后改 Cell 3 的 candidates 列表:
```python
import os
print("source 样本:", sorted(os.listdir(f'{WORK}/source'))[:10])
print("target 样本:", sorted(os.listdir(f'{WORK}/target_a'))[:10])
```

**Cell 4 input==gt** —— 配对错位。Cell 3 里 `link(... target_dir/tf, .../gt/sf)` 这一行的 `tf` (target 文件名) 必须跟 `sf` (source) 是同一张图的不同版本,不是同名两份。

**训练 OOM** —— 改小 batch (`config/default.yaml` 里 `train.batch`),或者 patch (`train.patch`,PPR10K 360p 图本身不大,patch=256 应该够)。